# 1.7 Structural Risk Minimization

Do not fix the model class in advance: search a ladder of nested classes, minimizing training error plus a complexity penalty.

This lesson operationalizes the bias-complexity tradeoff: lower rungs underfit, higher rungs overfit, and SRM minimizes training error plus a bound penalty. Generalization bounds supply the penalty; regularization turns the ladder into a continuous knob.

Save a copy to Drive to edit.

In [ ]:

import itertools
import math
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(16)


def theory_ladder(topic):
    """Return compact D1--D5 inputs for F16 theory notebooks."""
    if topic == "srm":
        return [
            {"name": "D1 lesson four-rung ladder", "emp": np.array([0.30, 0.15, 0.07, 0.05]), "H": np.array([2, 16, 256, 65536]), "m": 200, "delta": 0.05},
            {"name": "D2 six nested polynomial rungs", "emp": np.array([0.34, 0.21, 0.13, 0.09, 0.075, 0.07]), "H": np.array([2, 8, 32, 128, 512, 2048]), "m": 200, "delta": 0.05},
            {"name": "D3 larger sample shifts upward", "emp": np.array([0.30, 0.15, 0.07, 0.04, 0.035]), "H": np.array([2, 16, 256, 65536, 1048576]), "m": 2000, "delta": 0.05},
            {"name": "D4 noisy empirical errors", "emp": np.array([0.31, 0.19, 0.12, 0.115, 0.13]), "H": np.array([2, 16, 256, 4096, 65536]), "m": 300, "delta": 0.05},
            {"name": "D5 bad ordering stress case", "emp": np.array([0.28, 0.10, 0.03, 0.02, 0.00]), "H": np.array([4, 4096, 64, 1048576, 256]), "m": 120, "delta": 0.05},
        ]
    if topic == "regularization":
        x = np.linspace(-1.0, 1.0, 25)
        base = 1.0 + 2.0 * x - 1.5 * x ** 2
        return [
            {"name": "D1 lesson 3-row regression", "X": np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]]), "y": np.array([1.0, 2.0, 2.0]), "lambdas": np.array([0.0, 1.0, 10.0, 100.0])},
            {"name": "D2 dense lambda path", "X": np.column_stack([np.ones_like(x), x]), "y": 1.0 + 2.0 * x + 0.15 * np.sin(7.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D3 higher polynomial capacity", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3, x ** 4, x ** 5]), "y": base + 0.10 * np.sin(9.0 * x), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D4 noisy labels", "X": np.column_stack([np.ones_like(x), x, x ** 2, x ** 3]), "y": base + rng.normal(0.0, 0.18, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
            {"name": "D5 unscaled-feature trap", "X": np.column_stack([np.ones_like(x), x, 100.0 * x ** 2]), "y": base + rng.normal(0.0, 0.10, size=x.shape), "lambdas": np.logspace(-3, 2, 20)},
        ]
    if topic == "stability":
        sample = np.array([2.0, 4.0, 6.0, 8.0, 10.0])
        long_sample = np.linspace(0.0, 1.0, 50)
        return [
            {"name": "D1 lesson 5-number mean", "sample": sample, "kind": "mean"},
            {"name": "D2 m=50 bounded-range mean", "sample": long_sample, "kind": "mean"},
            {"name": "D3 ridge lambda ladder", "sample": np.column_stack([np.ones(20), np.linspace(-1.0, 1.0, 20)]), "target": 1.0 + np.linspace(-1.0, 1.0, 20), "lambdas": np.array([0.1, 0.3, 1.0, 3.0]), "kind": "ridge"},
            {"name": "D4 compare mean/ridge to 1-NN-style rule", "sample": np.linspace(0.0, 1.0, 20), "kind": "nn_compare"},
            {"name": "D5 outlier removal stress case", "sample": np.array([0.0, 0.1, 0.2, 0.3, 9.0]), "kind": "outlier"},
        ]
    if topic == "nfl":
        return [
            {"name": "D1 one unseen point", "n": 1, "prediction": np.array([0])},
            {"name": "D2 lesson three unseen points", "n": 3, "prediction": np.array([0, 0, 0])},
            {"name": "D3 four unseen points", "n": 4, "prediction": np.array([1, 0, 1, 0])},
            {"name": "D4 biased smooth subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "smooth"},
            {"name": "D5 adversarial mirror subset", "n": 4, "prediction": np.array([0, 0, 1, 1]), "subset": "mirror"},
        ]
    if topic == "uniform":
        return [
            {"name": "D1 one fixed hypothesis", "H": 1, "m": 500, "epsilon": 0.1},
            {"name": "D2 hundred-hypothesis lesson class", "H": 100, "m": 500, "epsilon": 0.1},
            {"name": "D3 low-m vacuous rungs", "H": 100, "ms": np.array([100, 150, 230, 300]), "epsilon": 0.1},
            {"name": "D4 solve m for delta", "H": 100, "delta": 0.05, "epsilon": 0.1},
            {"name": "D5 correlated non-iid violation", "H": 100, "m": 500, "epsilon": 0.1, "rho": 0.9},
        ]
    raise ValueError(topic)


def all_binary_targets(n):
    rows = list(itertools.product([0, 1], repeat=n))
    return np.array(rows, dtype=int)


def ridge_weights(X, y, lam, penalize_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    penalty = np.eye(X.shape[1])
    if not penalize_intercept:
        penalty[0, 0] = 0.0
    system = X.T @ X + lam * penalty
    rhs = X.T @ y
    weights = np.linalg.solve(system, rhs)
    return weights


def print_rows(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, item in enumerate(row):
            widths[i] = max(widths[i], len(str(item)))
    header = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(item).ljust(widths[i]) for i, item in enumerate(row)))


## The concept, built once (D1)

For each nested class $H_k$, SRM uses
$$\hat k=\arg\min_k\left[R_S(\hat h_k)+\sqrt{\frac{\ln|H_k|+\ln(1/\delta)}{2m}}\right].$$
The lesson's four rungs use $m=200$, $\delta=0.05$, empirical errors $0.30,0.15,0.07,0.05$, and class sizes $2,16,256,65536$.

In [ ]:

def srm_penalty(H_size, delta, m):
    numerator = np.log(H_size) + np.log(1.0 / delta)
    denominator = 2.0 * m
    penalty = np.sqrt(numerator / denominator)
    return penalty


def srm_select(emp_errors, H_sizes, delta, m):
    emp_errors = np.asarray(emp_errors, dtype=float)
    H_sizes = np.asarray(H_sizes, dtype=float)
    penalties = srm_penalty(H_sizes, delta, m)
    bounds = emp_errors + penalties
    best_index = int(np.argmin(bounds))
    return best_index, penalties, bounds


Now plug in the exact lesson numbers. The bound values should be $0.396,0.270,0.216,0.238$, so SRM selects rung $3$ even though rung $4$ has lower training error.

In [ ]:

lesson_emp = np.array([0.30, 0.15, 0.07, 0.05])
lesson_H = np.array([2, 16, 256, 65536])
lesson_best, lesson_penalties, lesson_bounds = srm_select(lesson_emp, lesson_H, 0.05, 200)
print_rows([(i + 1, f"{lesson_emp[i]:.3f}", f"{lesson_penalties[i]:.3f}", f"{lesson_bounds[i]:.3f}") for i in range(4)], ["rung", "emp", "penalty", "bound"])
assert np.allclose(np.round(lesson_penalties, 3), np.array([0.096, 0.120, 0.146, 0.188]))
assert np.allclose(np.round(lesson_bounds, 3), np.array([0.396, 0.270, 0.216, 0.238]))
assert lesson_best + 1 == 3


## The dataset ladder

D1 is the lesson ladder. D2 adds more nested rungs, D3 increases sample size, D4 adds noisy empirical errors, and D5 deliberately scrambles complexity ordering to stress the penalty.

In [ ]:

srm_ladder = theory_ladder("srm")
rows = []
for rung in srm_ladder:
    rows.append((rung["name"], len(rung["emp"]), int(rung["m"]), list(rung["H"][:3])))
print_rows(rows, ["dataset", "rungs", "m", "first H sizes"])


In [ ]:

srm_results = []
for dataset in srm_ladder:
    best, penalties, bounds = srm_select(dataset["emp"], dataset["H"], dataset["delta"], dataset["m"])
    srm_results.append({"dataset": dataset, "best": best, "penalties": penalties, "bounds": bounds})
    print(f"\n{dataset['name']}: selected rung {best + 1}")
    rows = []
    for i in range(len(dataset["emp"])):
        rows.append((i + 1, int(dataset["H"][i]), f"{dataset['emp'][i]:.3f}", f"{penalties[i]:.3f}", f"{bounds[i]:.3f}"))
    print_rows(rows, ["rung", "|H|", "emp", "penalty", "bound"])


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for j, result in enumerate(srm_results):
    dataset = result["dataset"]
    x_pos = np.arange(1, len(dataset["emp"]) + 1)
    axes[0, j].bar(x_pos, dataset["emp"], label="empirical")
    axes[0, j].bar(x_pos, result["penalties"], bottom=dataset["emp"], label="penalty")
    axes[0, j].axvline(result["best"] + 1, color="black", linestyle="--")
    axes[0, j].set_title(dataset["name"].split()[0])
    axes[0, j].set_xlabel("capacity rung")
    axes[0, j].set_ylim(0.0, max(result["bounds"]) * 1.2)
    axes[1, j].plot(x_pos, dataset["emp"], marker="o", label="emp")
    axes[1, j].plot(x_pos, result["penalties"], marker="o", label="penalty")
    axes[1, j].plot(x_pos, result["bounds"], marker="o", label="sum")
    axes[1, j].set_xlabel("capacity rung")
    axes[1, j].set_ylabel("SRM bound")
axes[0, 0].legend()
axes[1, 0].legend()
fig.suptitle("SRM capacity panels and bound-vs-capacity curves")
plt.tight_layout()


## Pitfall on D5: a penalty that misjudges complexity

If you sort the D5 ladder by its accidental display order instead of validated class size, the penalty understates some rich rungs and can choose an overfit rung. The fix is to use the correct $|H_k|$ bound term and a validated ladder.

In [ ]:

d5 = srm_ladder[-1]
wrong_H = np.array([4, 16, 64, 256, 1024])
wrong_best, wrong_penalties, wrong_bounds = srm_select(d5["emp"], wrong_H, d5["delta"], d5["m"])
right_best, right_penalties, right_bounds = srm_select(d5["emp"], d5["H"], d5["delta"], d5["m"])
print(f"Wrong penalty chooses rung {wrong_best + 1} with bound {wrong_bounds[wrong_best]:.3f}")
print(f"Correct penalty chooses rung {right_best + 1} with bound {right_bounds[right_best]:.3f}")
print(f"Naive ERM would choose rung {int(np.argmin(d5['emp'])) + 1}")


## Evaluate it + practice

- Metric: SRM bound by rung, reported as empirical error plus complexity penalty.
- No-skill baseline: naive ERM picks the minimum empirical error without a penalty.
- Cheap sanity check: penalties should increase with true class size and shrink when m grows.
- Ablation: turn off the penalty and watch D5 climb to the memorizing rung.
- Failure signals: selected rung is driven by accidental order or the bound is read as exact true risk.

### Practice prompts

- Change delta from 0.05 to 0.01 and predict which rung wins before plotting.

- Replace one class size with a larger value and explain the change.

- Add a validation-risk column and compare it with the SRM proxy.
